# Fraud Detection - Feature Engineering

This notebook covers:
- Feature Store setup with online-enabled FeatureViews
- Customer profile features (static, daily refresh)
- Aggregated transaction features (auto-managed Dynamic Table, 1-minute refresh)
- Derived risk features computed inline
- Training dataset generation via Feature Store spine join

### Import Libraries

Import Feature Store APIs for managing ML features and Snowpark for defining feature transformations.

In [ ]:
import sys
sys.path.append('../source')

from snowflake.snowpark import Session
from snowflake.snowpark.types import *
from snowflake.snowpark import functions as F
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, OnlineConfig
from snowflake.ml.feature_store.feature_view import FeatureViewStatus
import pandas as pd
import numpy as np

### Initialize Session

In [ ]:
import os
import snowflake.connector
from snowflake.snowpark import Session

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except:
    conn = snowflake.connector.connect(
        connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME")
    )
    session = Session.builder.configs({"connection": conn}).create()

session.sql("USE ROLE ACCOUNTADMIN").collect()
print(f"Role set to: {session.get_current_role()}")

### Set Database Context

Establish the working context for all subsequent operations. Using consistent database/schema across notebooks ensures all artifacts are created in the right location.

In [ ]:
DATABASE = "DEMO_FRAUD_DETECTION_DB"
SCHEMA = "ML_PIPELINE"
FEATURE_STORE_SCHEMA = "FEATURE_STORE"
WAREHOUSE = "ML_WH"

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()

## Initialize Feature Store

In [ ]:
fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=FEATURE_STORE_SCHEMA,
    default_warehouse=WAREHOUSE
)
print(f"Connected to Feature Store: {DATABASE}.{FEATURE_STORE_SCHEMA}")

## Create Customer Profile Feature View

### Examine Customer Profile Data

These **static features** (account age, credit score, fraud history) change infrequently but are critical for fraud risk assessment.

In [ ]:
customer_profiles_df = session.table(f"{DATABASE}.{SCHEMA}.CUSTOMER_PROFILES")
print(f"Customer profiles columns: {customer_profiles_df.columns}")
customer_profiles_df.show(5)

### Register Customer Profile Feature View

Create a Feature View for customer profile data:
- **refresh_freq='1 day'**: Profiles update daily (static data)
- **entities=[customer_entity]**: Join key is CUSTOMER_ID
- **version='V1'**: Enables tracking feature evolution

In [ ]:
customer_entity = fs.get_entity("CUSTOMER")

online_config = OnlineConfig(enable=True)

customer_profile_fv = FeatureView(
    name="CUSTOMER_PROFILE_FEATURES",
    entities=[customer_entity],
    feature_df=customer_profiles_df,
    refresh_freq="1 day",
    online_config=online_config,
    desc="Customer profile features for fraud detection - online enabled for real-time serving"
)

customer_profile_fv = fs.register_feature_view(
    feature_view=customer_profile_fv,
    version="V1",
    overwrite=True
)
print(f"Registered: {customer_profile_fv.name} (online enabled)")

## Create Aggregated Transaction Features

### Build Aggregated Transaction Features

We define the feature transformations using a **Snowpark DataFrame** — Snowflake's lazy, server-side DataFrame API. Snowpark DataFrames build a query plan that gets translated to SQL and executed on warehouse. 

**Note:** 

- Snowpark DataFrames execute on a warehouse (SQL engine), which is ideal for SQL-expressible transformations like aggregations and joins. 

- For heavier feature engineering that requires custom Python/ML logic at scale you can use **Container Runtime + Ray Data** on SPCS compute pools instead.

In [ ]:
transactions_df = session.table(f"{DATABASE}.{SCHEMA}.TRANSACTIONS")

agg_features_df = (
    transactions_df
    .filter(F.col("TRANSACTION_TIMESTAMP") >= F.dateadd("day", F.lit(-7), F.current_timestamp()))
    .group_by("CUSTOMER_ID")
    .agg(
        F.avg("TRANSACTION_AMOUNT").alias("D7_AVG_TXN_AMOUNT"),
        F.stddev("TRANSACTION_AMOUNT").alias("D7_STD_TXN_AMOUNT"),
        F.max("TRANSACTION_AMOUNT").alias("D7_MAX_TXN_AMOUNT"),
        F.min("TRANSACTION_AMOUNT").alias("D7_MIN_TXN_AMOUNT"),
        F.sum("TRANSACTION_AMOUNT").alias("D7_TOTAL_TXN_AMOUNT"),
        F.count(F.lit(1)).alias("D7_TXN_COUNT"),
        F.avg("TRANSACTION_HOUR").alias("D7_AVG_TXN_HOUR"),
        F.stddev("TRANSACTION_HOUR").alias("D7_STD_TXN_HOUR"),
        F.sum("IS_INTERNATIONAL").alias("D7_INTL_TXN_COUNT"),
        F.avg(F.col("IS_INTERNATIONAL").cast(FloatType())).alias("D7_INTL_TXN_RATIO"),
        F.avg(F.col("CARD_PRESENT").cast(FloatType())).alias("D7_CARD_PRESENT_RATIO")
    )
)

print(f"Aggregation features defined (7-day window): {agg_features_df.columns}")

### Register Aggregated Features as Feature View

Pass the transformation DataFrame directly to `FeatureView` — the Feature Store creates the backing Dynamic Table automatically:
- **refresh_freq='1 minute'**: New transactions trigger aggregate recomputation within 1 minute
- **online_config**: Enables low-latency point lookups for real-time inference
- Single refresh cadence (no DT + Feature View lag mismatch)

In [ ]:
agg_features_fv = FeatureView(
    name="CUSTOMER_AGG_FEATURES",
    entities=[customer_entity],
    feature_df=agg_features_df,
    refresh_freq="1 minute",
    online_config=online_config,
    desc="Aggregated transaction features per customer (7-day window, D7_ prefixed) - online enabled for real-time serving"
)

agg_features_fv = fs.register_feature_view(
    feature_view=agg_features_fv,
    version="V4",
    block=True,
    overwrite=True
)
print(f"Registered: {agg_features_fv.name} (version V4, online enabled)")

## Generate Training Dataset

### Create Spine DataFrame

The **spine** defines which entity instances to generate features for:
- Contains primary keys (TRANSACTION_ID, CUSTOMER_ID)
- One row per training example
- Feature Store joins features to this spine

**Why a spine?** It ensures we generate features only for the exact entities/timestamps we need, avoiding wasted computation.

In [ ]:
transactions_df = session.table(f"{DATABASE}.{SCHEMA}.TRANSACTIONS")
spine_df = transactions_df.select("TRANSACTION_ID", "CUSTOMER_ID")

print(f"Spine dataframe rows: {spine_df.count():,}")

### Generate Dataset from Feature Store

Use Feature Store's `generate_dataset()` to automatically join features to the spine:
- Combines multiple Feature Views in a single operation
- Handles join keys automatically based on entity definitions
- Creates a versioned, reproducible dataset

**Key benefit**: Training and serving use identical feature retrieval logic, preventing training-serving skew.

In [ ]:
training_data = fs.generate_dataset(
    name="FRAUD_TRAINING_DATASET",
    spine_df=spine_df,
    features=[
        fs.get_feature_view("CUSTOMER_PROFILE_FEATURES", "V1"),
        fs.get_feature_view("CUSTOMER_AGG_FEATURES", "V4")
    ],
    spine_timestamp_col=None,
    include_feature_view_timestamp_col=False,
    desc="Training dataset for fraud detection model (V4 - 7-day window, D7_ prefixed features)",
    output_type="dataset"
)

print(f"Generated training dataset")

### Combine All Features into Final Training Set

Join Feature Store data with:
1. **Transaction attributes**: Amount, time, card presence, etc.
2. **Labels**: IS_FRAUD (target variable)
3. **Derived features**: Computed inline from the joined data (risk ratios, anomaly flags)

This creates a comprehensive feature matrix with ~30 features capturing customer profiles, spending patterns, and transaction characteristics.

In [ ]:
training_df = training_data.read.to_snowpark_dataframe()
transactions_full = session.table(f"{DATABASE}.{SCHEMA}.TRANSACTIONS")

final_training_df = (
    training_df
    .join(
        transactions_full.select(
            "TRANSACTION_ID",
            "TRANSACTION_AMOUNT",
            "TRANSACTION_HOUR",
            "TRANSACTION_DAY_OF_WEEK",
            "IS_WEEKEND",
            "CARD_PRESENT",
            "IS_INTERNATIONAL",
            "IS_FRAUD"
        ),
        on="TRANSACTION_ID",
        how="inner"
    )
    .with_column("AMOUNT_TO_AVG_RATIO",
        F.when(F.col("D7_AVG_TXN_AMOUNT").is_null() | (F.col("D7_AVG_TXN_AMOUNT") == 0), F.lit(1.0))
         .otherwise(F.col("TRANSACTION_AMOUNT") / F.col("D7_AVG_TXN_AMOUNT")))
    .with_column("AMOUNT_TO_MAX_RATIO",
        F.when(F.col("D7_MAX_TXN_AMOUNT").is_null() | (F.col("D7_MAX_TXN_AMOUNT") == 0), F.lit(1.0))
         .otherwise(F.col("TRANSACTION_AMOUNT") / F.col("D7_MAX_TXN_AMOUNT")))
    .with_column("IS_HIGH_VALUE_TXN",
        F.when(F.col("D7_AVG_TXN_AMOUNT").is_null(), F.lit(0))
         .when(F.col("TRANSACTION_AMOUNT") > F.col("D7_AVG_TXN_AMOUNT") * 3, F.lit(1))
         .otherwise(F.lit(0)))
    .with_column("IS_ODD_HOUR",
        F.when((F.col("TRANSACTION_HOUR") < 6) | (F.col("TRANSACTION_HOUR") > 22), F.lit(1)).otherwise(F.lit(0)))
    .with_column("RISK_SCORE",
        F.when(F.col("D7_AVG_TXN_AMOUNT").is_null() | (F.col("D7_AVG_TXN_AMOUNT") == 0), F.lit(0.5))
         .otherwise(
            (F.col("TRANSACTION_AMOUNT") / F.col("D7_AVG_TXN_AMOUNT")) * 0.4
            + F.when((F.col("TRANSACTION_HOUR") < 6) | (F.col("TRANSACTION_HOUR") > 22), F.lit(1)).otherwise(F.lit(0)) * 0.2
            + (F.lit(1) - F.col("CARD_PRESENT")) * 0.2
            + F.col("IS_INTERNATIONAL") * 0.2))
    .na.fill({"D7_AVG_TXN_AMOUNT": 0.0, "D7_STD_TXN_AMOUNT": 0.0, "D7_MAX_TXN_AMOUNT": 0.0,
              "D7_MIN_TXN_AMOUNT": 0.0, "D7_TOTAL_TXN_AMOUNT": 0.0, "D7_TXN_COUNT": 0,
              "D7_AVG_TXN_HOUR": 0.0, "D7_STD_TXN_HOUR": 0.0, "D7_INTL_TXN_COUNT": 0,
              "D7_INTL_TXN_RATIO": 0.0, "D7_CARD_PRESENT_RATIO": 0.0})
)

print(f"Final training data columns: {final_training_df.columns}")
print(f"Total rows: {final_training_df.count():,}")

### Save Training Data

Persist the final training dataset as a Snowflake table. This enables:
- Reproducible model training across notebook sessions
- Data versioning and lineage tracking
- Efficient distributed training with Snowflake ML

In [ ]:
final_training_df.write.mode("overwrite").save_as_table(f"{DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA")
print(f"Saved training data to {DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA")

## Verify Feature Views

### Summary and Validation

Verify all Feature Views are registered and training data statistics:
- Confirm feature counts and data volumes
- Check class distribution (expect ~2% fraud)
- List all registered Feature Views for reference

In [ ]:
print("\n=== Feature Store Summary ===")
print("\nRegistered Feature Views:")
for fv in fs.list_feature_views().collect():
    print(f"  - {fv['NAME']} (version: {fv['VERSION']})")

print(f"\nTraining Data Statistics:")
train_df = session.table(f"{DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA")
print(f"  - Total samples: {train_df.count():,}")
print(f"  - Features: {len(train_df.columns) - 2}")
fraud_stats = train_df.group_by("IS_FRAUD").count().collect()
for row in fraud_stats:
    label = "Fraud" if row["IS_FRAUD"] == 1 else "Legitimate"
    print(f"  - {label}: {row['COUNT']:,}")